# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
# imports

import os
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from config import SYS_PARM_FIELDS_STRING

In [2]:
# Initialization

load_dotenv(override=True)

SERVICENOW_INSTANCE = os.getenv('SERVICENOW_INSTANCE')
SERVICENOW_USERNAME = os.getenv('SERVICENOW_USERNAME')
SERVICENOW_PASSWORD = os.getenv('SERVICENOW_PASSWORD')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
MODEL = "gpt-oss:latest"

openai = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

incidents_cache = []

assigned_to = "dgtes"

In [3]:
system_message = """
        You are an IT operations assistant. You have access to current ServiceNow incidents.
        When asked about incidents, use the provided incident data to answer questions.
        Keep responses concise and professional.
        """

In [ ]:
def get_incidents_servicenow():
    global incidents_cache

    url = f"https://{SERVICENOW_INSTANCE}/api/now/table/incident"
    headers = {"Content-Type":"application/json","Accept":"application/json"}
    params = {
        "sysparm_query": "stateNOT IN6,7,8^assigned_to.user_name="+assigned_to,
        "sysparm_limit": 10,
        "sysparm_fields": "assigned_to.user_name,number,caller_id.user_name,category,subcategory,cmdb_ci,state,impact,urgency,priorit,u_business_stopper,assignment_group.name,short_description,u_description,work_notes_list,comments"
    }
    try:
        response = requests.get(
            url,
            auth=(SERVICENOW_USERNAME, SERVICENOW_PASSWORD),
            headers=headers,
            params=params
        )
        if response.status_code != 200:
            raise Exception(f'Status:', response.status_code, 'Headers:', response.headers, 'Error Response:',response.json())
        
        incidents = response.json()['result']
        incidents_cache = incidents
        return incidents
    except Exception as e:
        return f"Error fetching incidents: {str(e)}"

In [8]:
def format_incidents_for_prompt(incidents):
    """Format incidents for prompt"""
    if not incidents:
        return "No incidents found."
    
    if isinstance(incidents, str):  # Error message
        return incidents
    
    formatted = []
    for incident in incidents[:10]:  # Limit to 10 incidents
        formatted.append({
            "number": incident.get('number', 'N/A'),
            "short_description": incident.get('short_description', 'No description'),
            "priority": incident.get('priority', 'N/A'),
            "assigned_to": incident.get('assigned_to.user_name', 'Unassigned'),
            "opened_at": incident.get('opened_at', 'N/A')
        })
    
    return json.dumps(formatted, indent=2)

In [9]:
def get_incidents():
    if not incidents_cache:
        incidents = get_incidents_servicenow()
        if isinstance(incidents, list):
            incidents_data = format_incidents_for_prompt(incidents)
        else:
            incidents_data = incidents  # Error message
    else:
        incidents_data = format_incidents_for_prompt(incidents_cache)
    return incidents_data

In [11]:
get_incidents()

'[\n  {\n    "number": "INC0296300",\n    "short_description": "Service Now development",\n    "priority": "4",\n    "assigned_to": "DGTES",\n    "opened_at": "N/A"\n  },\n  {\n    "number": "INC0296930",\n    "short_description": "test",\n    "priority": "4",\n    "assigned_to": "DGTES",\n    "opened_at": "N/A"\n  },\n  {\n    "number": "INC0287164",\n    "short_description": "2 survey values on 1 IT ticket",\n    "priority": "4",\n    "assigned_to": "DGTES",\n    "opened_at": "N/A"\n  },\n  {\n    "number": "INC0286041",\n    "short_description": "Performance analitycs graph",\n    "priority": "4",\n    "assigned_to": "DGTES",\n    "opened_at": "N/A"\n  },\n  {\n    "number": "INC0289438",\n    "short_description": "Failed to create External Teams Site \\"WUZHOU-ACARIZAX\\"",\n    "priority": "4",\n    "assigned_to": "DGTES",\n    "opened_at": "N/A"\n  },\n  {\n    "number": "INC0296314",\n    "short_description": "PTSK0002668 task",\n    "priority": "4",\n    "assigned_to": "DGTES",

In [12]:
get_incidents_function = {
    "name": "get_incidents",
    "description": "Retrieve ServiceNow incidents.",
    "parameters": {
        "type": "object",
        #"properties": {
        #    "destination_city": {
        #        "type": "string",
        #        "description": "The city that the customer wants to travel to",
        #    },
        #},
        #"required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": get_incidents_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_incidents',
   'description': 'Retrieve ServiceNow incidents.',
   'parameters': {'type': 'object', 'additionalProperties': False}}}]

In [13]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [14]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_incidents":
            #arguments = json.loads(tool_call.function.arguments)
            #city = arguments.get('destination_city')
            details = get_incidents()
            responses.append({
                "role": "tool",
                "content": details,
                "tool_call_id": tool_call.id
            })
    return responses

In [15]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A bit more about what Gradio actually does:

1. Gradio constructs a frontend Svelte app based on our Python description of the UI
2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this React app
3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

And of course when Gradio generates the frontend app, it ensures that the the Submit button calls the right backend route.

That's it!

It's simple, and it has a result that feels magical.

# Let's go multi-modal!!

We can use DALL-E-3, the image generation model behind GPT-4o, to make us some images

Let's put this in a function called artist.

### Price alert: each time I generate an image it costs about 4 cents - don't go crazy with images!

In [ ]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [ ]:
def artist(city):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
image = artist("New York City")
display(image)

In [ ]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

## Let's bring this home:

1. A multi-modal AI assistant with image and audio generation
2. Tool callling with database lookup
3. A step towards an Agentic workflow


In [ ]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [ ]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## The 3 types of Gradio UI

`gr.Interface` is for standard, simple UIs

`gr.ChatInterface` is for standard ChatBot UIs

`gr.Blocks` is for custom UIs where you control the components and the callbacks

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

# Exercises and Business Applications

Add in more tools - perhaps to simulate actually booking a flight. A student has done this and provided their example in the community contributions folder.

Next: take this and apply it to your business. Make a multi-modal AI assistant with tools that could carry out an activity for your work. A customer support assistant? New employee onboarding assistant? So many possibilities! Also, see the week2 end of week Exercise in the separate Notebook.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a HUGE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>